In [1]:
from jupyterlab.utils import deprecated
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import warnings
import sys
sys.path.append('../')
from src import *

seed = 42
np.random.seed(seed)
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_parquet("../data/shuffled.parquet")

In [4]:
# df.iloc[1_000_000:].to_parquet(
#             '../data/all_raman_spectra.parquet',
#             compression='zstd',      # лучшее сжатие
#             index=False
#         )

In [5]:
X, y = df[['X', 'Y', 'Wave', 'Intensity', 'group', 'center', 'brain']], df['label']
# df_shuffled.to_parquet(
#     '/home/noda/Projects/raman-spectroscopy/data/shuffled.parquet',
#     compression='zstd',      # лучшее сжатие
#     index=False
# )

In [6]:
del df
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
del X
del y

KeyboardInterrupt: 

In [ ]:
type(X_train)

In [ ]:
from sklearn.metrics import f1_score
import joblib

X_train = X_train.drop(columns=['X', 'Y', 'group'], errors='ignore')
X_test = X_test.drop(columns=['X', 'Y', 'group'], errors='ignore')


model = CatBoostClassifier(
    iterations=2000,
    task_type="GPU",
    devices='0',
    loss_function='MultiClass',
    early_stopping_rounds=100,
    verbose=100,
    depth=7,
    random_seed=seed,
    learning_rate=0.03,
    subsample=0.6,
    l2_leaf_reg=5,
    bootstrap_type='Bernoulli',
    one_hot_max_size=10
)

model.fit(X_train, y_train, cat_features=['center', 'brain'])
joblib.dump(model, '../models/model-catboost(3).joblib')

y_pred = model.predict(X_test)

score = f1_score(y_test, y_pred, average='macro')
print(score)

In [ ]:
import matplotlib.pyplot as plt

# Получаем важность признаков
feature_importance = model.get_feature_importance(type='FeatureImportance')
feature_names = X_train.columns

# Сортируем для визуализации
sorted_idx = np.argsort(feature_importance)

plt.figure(figsize=(10, 6))
plt.barh(range(len(sorted_idx)), feature_importance[sorted_idx], align='center')
plt.yticks(range(len(sorted_idx)), [feature_names[i] for i in sorted_idx])
plt.title('CatBoost Feature Importance')
plt.xlabel('Importance Score')
plt.show()

# Выведем значения текстом для точности
for name, score in zip(feature_names, feature_importance):
    print(f"{name}: {score:.4f}")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report, f1_score

def get_scores(y_test, y_pred):
    # Accuracy — общая метрика, average здесь не нужен
    acc = accuracy_score(y_test, y_pred)

    # Для многоклассовой классификации используем average='macro' или 'weighted'
    f1 = f1_score(y_test, y_pred, average='macro')
    prec = precision_score(y_test, y_pred, average='macro')
    rec = recall_score(y_test, y_pred, average='macro')

    print(f'F1 (Macro) = {f1:.4f}')
    print(f'Acc = {acc:.4f} | Prec = {prec:.4f} | Recall = {rec:.4f}')

    # Classification report сам выводит точность по каждому из 3 классов
    print('\nDetailed Classification Report:')
    print(classification_report(y_test, y_pred))

get_scores(y_test, y_pred)
